## Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.join(os.path.dirname('.'), 'src'))

from simulation import simulate_gbm
from analytics import calculate_mean, calculate_variance, calculate_var
from visualization import plot_price_paths, plot_distribution, plot_convergence

# Set matplotlib backend for notebook
%matplotlib inline

# Simulation parameters
S0 = 100  # Initial price
mu = 0.05  # Drift (5% annual return)
T = 1  # Time horizon (1 year)
dt = 0.01  # Time step

## Monte Carlo Simulation Engine

We implement Geometric Brownian Motion (GBM) to simulate stock price paths:

$$ dS = \mu S dt + \sigma S dW $$

Where:
- $S$: Stock price
- $\mu$: Drift coefficient
- $\sigma$: Volatility
- $dW$: Wiener process increment

In [ ]:
# Example simulation
sigma = 0.2  # 20% volatility
num_simulations = 1000

paths = simulate_gbm(S0, mu, sigma, T, dt, num_simulations)
print(f"Generated {num_simulations} simulation paths")
print(f"Path shape: {paths.shape}")
print(f"Initial price: {paths[0, 0]}")
print(f"Sample final prices: {paths[:5, -1]}")

## Compute Analytics Metrics

Calculate key statistical measures and risk metrics from the simulated price paths.

In [ ]:
final_prices = paths[:, -1]

mean_price = calculate_mean(final_prices)
variance = calculate_variance(final_prices)
var_95 = calculate_var(final_prices, confidence=0.95)

print(f"Mean final price: ${mean_price:.2f}")
print(f"Variance: {variance:.2f}")
print(f"VaR (95% confidence): ${var_95:.2f}")
print(f"Expected price (theoretical): ${S0 * np.exp(mu * T):.2f}")

## Plot Price Paths, Distribution, and Convergence

Generate the three core visualizations to analyze simulation behavior.

In [ ]:
# Price paths
plt.figure(figsize=(12, 6))
for i in range(min(20, num_simulations)):
    plt.plot(paths[i], alpha=0.6)
plt.title('Sample Price Paths (σ = 0.2)')
plt.xlabel('Time Steps')
plt.ylabel('Price')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Distribution
plt.figure(figsize=(10, 6))
plt.hist(final_prices, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(mean_price, color='red', linestyle='--', label=f'Mean: ${mean_price:.2f}')
plt.axvline(var_95, color='orange', linestyle='--', label=f'VaR 95%: ${var_95:.2f}')
plt.title('Final Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Convergence analysis
simulation_counts = list(range(100, 5100, 200))
means_conv = []
vars_conv = []

for n_sim in simulation_counts:
    paths_conv = simulate_gbm(S0, mu, sigma, T, dt, n_sim)
    final_prices_conv = paths_conv[:, -1]
    means_conv.append(calculate_mean(final_prices_conv))
    vars_conv.append(calculate_var(final_prices_conv))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(simulation_counts, means_conv, marker='o')
ax1.set_title('Mean Convergence')
ax1.set_xlabel('Number of Simulations')
ax1.set_ylabel('Mean Price')
ax1.grid(True, alpha=0.3)

ax2.plot(simulation_counts, vars_conv, marker='o', color='green')
ax2.set_title('VaR Convergence (95%)')
ax2.set_xlabel('Number of Simulations')
ax2.set_ylabel('VaR')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Experimental Parameter Sweep

Run systematic experiments across different volatility levels and simulation sizes.

In [ ]:
volatilities = [0.1, 0.2, 0.4]
simulation_sizes = [500, 1000, 2000]

results = []

for vol in volatilities:
    for n_sim in simulation_sizes:
        paths_exp = simulate_gbm(S0, mu, vol, T, dt, n_sim)
        final_prices_exp = paths_exp[:, -1]
        
        mean_val = calculate_mean(final_prices_exp)
        var_val = calculate_variance(final_prices_exp)
        va_r = calculate_var(final_prices_exp)
        
        results.append({
            'volatility': vol,
            'simulations': n_sim,
            'mean': mean_val,
            'variance': var_val,
            'VaR_95': va_r
        })

# Display results table
import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

## Results Analysis and Observations

### Key Findings

1. **Convergence Behavior**: Monte Carlo estimates stabilize as simulation count increases, with VaR requiring more simulations than mean estimates for convergence.

2. **Volatility Impact**: Higher volatility leads to wider price distributions and increased uncertainty in risk estimates.

3. **Minimum Simulations**: For reliable risk estimation, at least 1000-2000 simulations are typically required, especially under high volatility conditions.

4. **Risk Metrics**: Value at Risk (VaR) provides a conservative estimate of potential losses, becoming more stable with larger simulation sets.

### Practical Implications

- Small simulation sets (< 500) produce noisy and unreliable estimates
- High volatility environments require larger simulation counts for accurate risk assessment
- Monte Carlo methods are computationally intensive but provide robust risk estimates when properly configured